# Finalisation pipeline

Builds the final schema **`graph_db`** (`dbdiagrams/final_dbdiagram.txt`) from the cleaned tables in `clean_up_1`:
only the columns the analysis needs, with real primary and foreign keys, so every holding is an edge between an
existing owner (`edgar_company`) and an existing security (`asset`).

| `graph_db` table | from `clean_up_1` | kept |
|---|---|---|
| `ff_industry` | `ff_industry` | all rows and columns |
| `sic_ff` | `sic_ff` | all rows and columns |
| `edgar_company` | `edgar_company` | `cik`, `name` (current EDGAR name), `sic`, `sic_description`, `is_owner`, `is_13f_filer` (all rows; `is_owner` recomputed against `graph_db.holdings_clean`) |
| `asset` | `asset` | `cusip`, `issuer_name`, `issuer_description`, `figi_name`, `market_sector`, `security_type2`, `issuer_cik`, `industry_match` (all rows, also assets without an EDGAR issuer) |
| `holdings_clean` | `holdings_clean` | one row per (`cik`, `ryear`, `rquarter`, `cusip`) with a synthetic key `holding_id`: only the latest report of an owner in a quarter, sub-positions summed (`value`, `shares`), latest `fdate`, `split`; only holdings whose CUSIP is in `asset` |

`clean_up_1.edgar_company_name` (former names) is not copied: it is only needed for the issuer matching of
`5_industry_pipeline.ipynb`; the current name is `edgar_company.name`.

Foreign keys: `holdings_clean.cusip → asset`, `holdings_clean.cik → edgar_company`,
`asset.issuer_cik → edgar_company` (NULL allowed: asset without EDGAR issuer), `sic_ff → ff_industry`.
`edgar_company.sic → sic_ff` stays a logical join (one `sic_ff` row per scheme).

Run after all other pipelines (`README.md`). Every block shows the database feedback and is written to the run log
`logs/finalisation/<RUN_ID>/`, so runs can be compared later (section 8). Execution and logging code lives in
`src/pipeline_helpers.py`.

| Section | Phase label | What happens |
|---|---|---|
| 0.2 Database connection | `connection` | read-only check, then working connection |
| 1. Schema | `final_schema` | `graph_db`, previous tables dropped |
| 2. Industry tables | `final_industry` | `ff_industry`, `sic_ff` |
| 3. Company table | `final_company` | `edgar_company` |
| 4. Asset table | `final_asset` | `asset` |
| 5. Holdings table | `final_holdings` | `holdings_clean` (~102M rows to ~68M, ~15–25 min), keys, indexes, foreign keys, `is_owner` |
| 6. Graph views | `final_views` | `mv_owner_asset_weight` (owner → asset edges with portfolio weights), indexes |
| 7. Checks | `final_checks`, `rdate_analysis` | row counts against `clean_up_1`, dropped holdings, totals, report dates within a quarter, keys, sizes |

> **The cells modify the database.** All `graph_db` tables and views are dropped and rebuilt; `clean_up_1` is only read.

## 0. Setup

### 0.1 Configuration

* `SRC`, `DST`: source and target schema

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent                       # .../Database/Initialisation (notebooks run from notebooks/)
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "config")]

from pipeline_helpers import PipelineRun, compare_runs, expected_mismatches, load_logs, phase_durations, steps_table
from local_config import DSN   # machine-specific, git-ignored (template: local_config.example.py)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

LOG_ROOT = ROOT / "logs" / "finalisation"

SRC = "clean_up_1"
DST = "graph_db"

TABLES = ["ff_industry", "sic_ff", "edgar_company", "asset", "holdings_clean"]
# no longer part of graph_db: dropped if an earlier run built them
OLD_TABLES = ["edgar_company_name"]
# materialized views on top of the tables (section 6)
VIEWS = ["mv_owner_asset_weight"]

# holdings_clean has ~100M rows: pg_class estimates instead of count(*) for it (row_estimate)
p = PipelineRun(LOG_ROOT, dsn=DSN, max_rows=1000, exact_counts=False)
print("RUN_ID :", p.run_id)
print("LOG_DIR:", p.log_dir)

RUN_ID : 20261029_160620
LOG_DIR: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/logs/finalisation/20261029_160620


### 0.2 Database connection

Read-only check first (`READ ONLY` transaction, rolled back). All five `clean_up_1` tables must exist (built by the
other pipelines). The working connection (autocommit) is opened only if the check passes.

In [2]:
CONNECTION_OK = p.check_connection(DSN, schemas=[SRC],
                                   tables=[f"{SRC}.{t}" for t in TABLES] + [f"{DST}.{t}" for t in TABLES])

✅ **[connection] connection_check_server**: `ok` | 0.0s

> READ ONLY transaction, rolled back. Nothing committed.

,server_version,database,db_user,is_standby,txn_read_only,db_size,other_sessions
0,PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04....,Holdings_v2,postgres,False,on,70 GB,0


✅ **[connection] connection_check_schemas**: `ok`

,schema_name,schema_exists,can_create,can_use
0,clean_up_1,True,True,True


✅ **[connection] connection_check_tables**: `ok`

> Existing tables will be dropped and recreated by the pipeline.

,table_name,exists,unlogged,est_rows,total_size
0,clean_up_1.edgar_company,True,False,88009,19 MB
1,clean_up_1.asset,True,False,40593,22 MB
2,clean_up_1.ff_industry,True,False,209,64 kB
3,clean_up_1.sic_ff,True,False,59627,4912 kB
4,graph_db.ff_industry,True,False,209,64 kB
5,graph_db.sic_ff,True,False,59627,4912 kB
6,graph_db.edgar_company,True,False,88009,10 MB
7,graph_db.asset,True,False,40593,6096 kB
8,graph_db.holdings_clean,True,False,67649392,9918 MB
9,clean_up_1.holdings_clean,True,False,97681472,25 GB


In [3]:
assert CONNECTION_OK, "Connection check failed, see connection_check_server above"
p.connect()

Connected: user=postgres password=xxx dbname=Holdings_v2 host=localhost port=5432


## 1. Schema

`graph_db` is created if missing; its tables are dropped (`CASCADE` drops the foreign keys between them) and rebuilt
in the order of their references: industries, companies, assets, holdings. `OLD_TABLES` (no longer part of
`graph_db`) are dropped too.

In [4]:
sch = p.stage("final_schema")

sch.run("create_schema", f"CREATE SCHEMA IF NOT EXISTS {DST};")
sch.run("drop_tables", f"DROP TABLE IF EXISTS {', '.join(f'{DST}.{t}' for t in OLD_TABLES + TABLES[::-1])} CASCADE;")

✅ **[final_schema] create_schema**: `ok` | `CREATE SCHEMA` | 0.0s

> NOTICES: NOTICE:  schema "graph_db" already exists, skipping

✅ **[final_schema] drop_tables**: `ok` | `DROP TABLE` | 0.1s

> NOTICES: NOTICE:  table "edgar_company_name" does not exist, skipping NOTICE:  drop cascades to materialized view graph_db.mv_owner_asset_weight

""


## 2. Industry tables

Copied as they are (`5_industry_pipeline.ipynb`). Industry of a company in any scheme:
`edgar_company.sic → sic_ff (scheme = …) → ff_industry`.

In [5]:
ind = p.stage("final_industry")

ind.run("create_ff_industry", f"""
    CREATE TABLE {DST}.ff_industry (
        scheme             SMALLINT NOT NULL CHECK (scheme IN (5, 10, 12, 17, 30, 38, 48, 49)),
        industry_number    SMALLINT NOT NULL,
        short_name         TEXT     NOT NULL,
        industry_name      TEXT     NOT NULL,
        residual_category  BOOLEAN  NOT NULL,
        PRIMARY KEY (scheme, industry_number)
    );

    INSERT INTO {DST}.ff_industry
    SELECT scheme, industry_number, short_name, industry_name, residual_category
    FROM {SRC}.ff_industry;
""", expected=209)

ind.run("create_sic_ff", f"""
    CREATE TABLE {DST}.sic_ff (
        sic              VARCHAR(4) NOT NULL CHECK (sic ~ '^[0-9]{{4}}$'),
        scheme           SMALLINT   NOT NULL,
        industry_number  SMALLINT   NOT NULL,
        PRIMARY KEY (sic, scheme),
        FOREIGN KEY (scheme, industry_number) REFERENCES {DST}.ff_industry (scheme, industry_number)
    );

    INSERT INTO {DST}.sic_ff
    SELECT sic, scheme, industry_number
    FROM {SRC}.sic_ff;
""", expected=59_627)
ind.run("idx_sic_ff_industry", f"CREATE INDEX sic_ff_industry_idx ON {DST}.sic_ff (scheme, industry_number);")

✅ **[final_industry] create_ff_industry**: `ok` | `INSERT 0 209` | rows: **209** | expected: 209 (diff +0) | 0.0s

✅ **[final_industry] create_sic_ff**: `ok` | `INSERT 0 59627` | rows: **59,627** | expected: 59,627 (diff +0) | 0.2s

✅ **[final_industry] idx_sic_ff_industry**: `ok` | `CREATE INDEX` | 0.0s

""


## 3. Company table

`edgar_company` keeps the identity, the current name and the SIC of every EDGAR company (owners and issuers).
`is_owner` is copied here and recomputed in section 5 against `graph_db.holdings_clean`. Index on `sic` for the
industry joins; `name` is only read in full scans, so it has no index.

In [6]:
com = p.stage("final_company")

com.run("create_edgar_company", f"""
    CREATE TABLE {DST}.edgar_company (
        cik              INTEGER    PRIMARY KEY,
        name             TEXT,
        sic              VARCHAR(4),
        sic_description  TEXT,
        is_owner         BOOLEAN    NOT NULL,
        is_13f_filer    BOOLEAN    NOT NULL
    );

    INSERT INTO {DST}.edgar_company (cik, name, sic, sic_description, is_owner, is_13f_filer)
    SELECT cik, name, sic, sic_description, is_owner, is_13f_filler
    FROM {SRC}.edgar_company;
""", expected=88_009)
com.run("idx_edgar_company_sic", f"CREATE INDEX edgar_company_sic_idx ON {DST}.edgar_company (sic);")

✅ **[final_company] create_edgar_company**: `ok` | `INSERT 0 88009` | rows: **88,009** | expected: 88,009 (diff +0) | 0.1s

✅ **[final_company] idx_edgar_company_sic**: `ok` | `CREATE INDEX` | 0.1s

""


## 4. Asset table

One row per CUSIP with the issuer link. Assets without an EDGAR issuer (funds / ETFs, unmatched or ambiguous names)
are kept with `issuer_cik` `NULL`; the foreign key allows that.

In [7]:
ast = p.stage("final_asset")

ast.run("create_asset", f"""
    CREATE TABLE {DST}.asset (
        cusip               VARCHAR(9) PRIMARY KEY,
        issuer_name         TEXT,
        issuer_description  TEXT,
        figi_name           TEXT,
        market_sector       TEXT,
        security_type2      TEXT,
        issuer_cik          INTEGER    REFERENCES {DST}.edgar_company (cik),
        industry_match      TEXT
    );

    INSERT INTO {DST}.asset (cusip, issuer_name, issuer_description, figi_name, market_sector, security_type2,
                             issuer_cik, industry_match)
    SELECT cusip, issuer_name, issuer_description, figi_name, market_sector, security_type2,
           issuer_cik, industry_match
    FROM {SRC}.asset;
""", expected=40_593)
ast.run("idx_asset_issuer_cik", f"CREATE INDEX asset_issuer_cik_idx ON {DST}.asset (issuer_cik);")

✅ **[final_asset] create_asset**: `ok` | `INSERT 0 40593` | rows: **40,593** | expected: 40,593 (diff +0) | 0.2s

✅ **[final_asset] idx_asset_issuer_cik**: `ok` | `CREATE INDEX` | 0.0s

""


## 5. Holdings table

Only holdings whose CUSIP is in `graph_db.asset` are copied (the others have no security node; ~37k rows, 0.04%).

**Report quarter instead of report date.** `rdate` becomes `ryear` + `rquarter` (the quarter `rdate` falls in).
99.04% of the holdings are on a quarter end and 81% of the rest within its last week (28th–30th); the others are
mid-quarter reports. 433 owners filed two or more reports with different `rdate`s in one quarter (528
owner-quarters): the same report twice, or a mid-quarter report followed by the regular quarter-end one (section 7,
*Report dates within a quarter*). Summing them would count a position twice, so **only the latest report of an owner
in a quarter is kept** (`latest_report`); in 519 of the 528 cases that is the quarter-end report. Dropped: ~166k
holdings lines of the earlier reports (0.12% of value).

**One row per owner, quarter and security.** A 13F filing often lists the same CUSIP several times: one line per
investment discretion, voting authority or other manager (in `clean_up_1` 13.9M (`cik`, `rdate`, `cusip9`) groups
have more than one row, 34M extra rows); a few come from an original filing plus a NEW HOLDINGS amendment. They are
summed into one position:

* `value`, `shares`: sum of the sub-positions (none of the sums is 0, no negative rows, so the `CHECK`s still hold)
* `fdate`: latest filing date of the sub-positions
* `split`: `bool_or` (the sub-positions never disagree)

`(cik, ryear, rquarter, cusip)` is `UNIQUE`; `holding_id` (`BIGINT` identity) is the primary key, numbered in
(`cik`, `ryear`, `rquarter`, `cusip`) order so it is the same in every run on the same data.

The table is filled first and gets its keys, indexes and foreign keys afterwards (much faster than checking ~68M
rows one by one). The unique index starts with `cik`, so it also serves lookups by owner (no separate `cik` index).

Takes about 15–25 minutes. Interrupting the kernel (■) stops the current block; re-run the section from the top.

In [8]:
hld = p.stage("final_holdings")

hld.run("create_holdings", f"""
    CREATE TABLE {DST}.holdings_clean (
        holding_id  BIGINT         GENERATED ALWAYS AS IDENTITY,
        cik         INTEGER        NOT NULL,
        ryear       SMALLINT       NOT NULL,
        rquarter    SMALLINT       NOT NULL CHECK (rquarter BETWEEN 1 AND 4),
        fdate       DATE           NOT NULL,
        cusip       CHAR(9)        NOT NULL,
        value       NUMERIC(20,5)  NOT NULL,
        shares      NUMERIC(21,3)  NOT NULL,
        split       BOOLEAN,
        CONSTRAINT chk_value_valid CHECK (
            value <> 0 AND value NOT IN ('Infinity'::numeric, '-Infinity'::numeric, 'NaN'::numeric)),
        CONSTRAINT chk_shares_valid CHECK (
            shares <> 0 AND shares NOT IN ('Infinity'::numeric, '-Infinity'::numeric, 'NaN'::numeric))
    );
""")

# the GROUP BY / ORDER BY of ~100M rows: more memory per sort / hash (this session only)
hld.run("work_mem", "SET work_mem = '1GB';")

hld.run("latest_report", f"""
    -- latest report date of every owner and quarter (session temp table, used by insert_holdings)
    DROP TABLE IF EXISTS pg_temp.latest_report;
    CREATE TEMP TABLE latest_report AS
    SELECT h.cik, max(h.rdate) AS rdate
    FROM {SRC}.holdings_clean h
    WHERE EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9)
    GROUP BY h.cik, date_trunc('quarter', h.rdate);
""", expected=276_423, note="owner-quarters (~2 min)")

hld.run("insert_holdings", f"""
    INSERT INTO {DST}.holdings_clean (cik, ryear, rquarter, cusip, fdate, value, shares, split)
    SELECT h.cik,
           extract(year    FROM h.rdate)::smallint,
           extract(quarter FROM h.rdate)::smallint,
           h.cusip9, max(h.fdate), sum(h.value), sum(h.shares), bool_or(h.split)
    FROM {SRC}.holdings_clean h
    JOIN pg_temp.latest_report l USING (cik, rdate)
    WHERE EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9)
    GROUP BY h.cik, h.rdate, h.cusip9      -- one rdate per owner and quarter: same groups as (cik, ryear, rquarter, cusip)
    ORDER BY h.cik, h.rdate, h.cusip9;
""", expected=67_649_392, note="one row per (cik, ryear, rquarter, cusip) of the 101,611,848 holdings lines in the "
                               "latest reports whose CUSIP is in asset")

hld.run("drop_latest_report", "DROP TABLE pg_temp.latest_report;")
hld.run("reset_work_mem", "RESET work_mem;")

✅ **[final_holdings] create_holdings**: `ok` | `CREATE TABLE` | 0.0s

✅ **[final_holdings] work_mem**: `ok` | `SET` | 0.0s

✅ **[final_holdings] latest_report**: `ok` | `SELECT 276848` | rows: **276,848** | expected: 276,423 (diff +425) | 79.3s

> owner-quarters (~2 min)

NOTICES: NOTICE:  schema "pg_temp" does not exist, skipping

✅ **[final_holdings] insert_holdings**: `ok` | `INSERT 0 67299418` | rows: **67,299,418** | expected: 67,649,392 (diff -349,974) | 382.4s

> one row per (cik, ryear, rquarter, cusip) of the 101,611,848 holdings lines in the latest reports whose CUSIP is in asset

✅ **[final_holdings] drop_latest_report**: `ok` | `DROP TABLE` | 0.0s

✅ **[final_holdings] reset_work_mem**: `ok` | `RESET` | 0.0s

""


In [9]:
hld.run("pk_holdings", f"""
    ALTER TABLE {DST}.holdings_clean ADD CONSTRAINT pk_holdings PRIMARY KEY (holding_id);
""")
hld.run("uq_holdings_cik_ryear_rquarter_cusip", f"""
    ALTER TABLE {DST}.holdings_clean
        ADD CONSTRAINT uq_holdings_cik_ryear_rquarter_cusip UNIQUE (cik, ryear, rquarter, cusip);
""")
hld.run("idx_holdings_cusip", f"CREATE INDEX idx_holdings_cusip ON {DST}.holdings_clean (cusip);")
hld.run("idx_holdings_ryear_rquarter",
        f"CREATE INDEX idx_holdings_ryear_rquarter ON {DST}.holdings_clean (ryear, rquarter);")

hld.run("fk_holdings_asset", f"""
    ALTER TABLE {DST}.holdings_clean
        ADD CONSTRAINT fk_holdings_asset FOREIGN KEY (cusip) REFERENCES {DST}.asset (cusip);
""")
hld.run("fk_holdings_company", f"""
    ALTER TABLE {DST}.holdings_clean
        ADD CONSTRAINT fk_holdings_company FOREIGN KEY (cik) REFERENCES {DST}.edgar_company (cik);
""")

✅ **[final_holdings] pk_holdings**: `ok` | `ALTER TABLE` | 94.5s

✅ **[final_holdings] uq_holdings_cik_ryear_rquarter_cusip**: `ok` | `ALTER TABLE` | 111.3s

✅ **[final_holdings] idx_holdings_cusip**: `ok` | `CREATE INDEX` | 77.7s

✅ **[final_holdings] idx_holdings_ryear_rquarter**: `ok` | `CREATE INDEX` | 68.1s

✅ **[final_holdings] fk_holdings_asset**: `ok` | `ALTER TABLE` | 12.2s

✅ **[final_holdings] fk_holdings_company**: `ok` | `ALTER TABLE` | 11.0s

""


`is_owner`: `true` if the CIK has holdings in `graph_db.holdings_clean`. Owners whose holdings were all dropped
(CUSIPs not in `asset`) become `false`.

In [10]:
hld.run("update_is_owner", f"""
    UPDATE {DST}.edgar_company c
    SET is_owner = EXISTS (SELECT 1 FROM {DST}.holdings_clean h WHERE h.cik = c.cik)
    WHERE c.is_owner IS DISTINCT FROM EXISTS (SELECT 1 FROM {DST}.holdings_clean h WHERE h.cik = c.cik);
""", note="rows changed: owners whose holdings were all dropped")

for t in TABLES:
    hld.run(f"analyze_{t}", f"ANALYZE {DST}.{t};")

✅ **[final_holdings] update_is_owner**: `ok` | `UPDATE 0` | rows: **0** | 2.5s

> rows changed: owners whose holdings were all dropped

✅ **[final_holdings] analyze_ff_industry**: `ok` | `ANALYZE` | 0.0s

✅ **[final_holdings] analyze_sic_ff**: `ok` | `ANALYZE` | 0.0s

✅ **[final_holdings] analyze_edgar_company**: `ok` | `ANALYZE` | 0.1s

✅ **[final_holdings] analyze_asset**: `ok` | `ANALYZE` | 0.1s

✅ **[final_holdings] analyze_holdings_clean**: `ok` | `ANALYZE` | 0.9s

## 6. Graph views

### Owner → asset (CUSIP)

`mv_owner_asset_weight`: one edge per (`owner`, `asset`) and quarter of the bipartite owner–security graph.
`weight` = share of the position in the owner's portfolio value of that quarter, so the weights of an owner sum to 1
per quarter. Positions with `value` 0 are left out (none at present: `chk_value_valid`).

Rows are stored in quarter order, so reading one quarter touches few pages. The unique index
(`ryear`, `rquarter`, `owner`, `asset`) is required for `REFRESH MATERIALIZED VIEW CONCURRENTLY` and serves
per-quarter and owner lookups; the second index serves asset lookups. Section 1 drops the view together with the
tables (`CASCADE`); it is rebuilt here. ~5–10 minutes.

In [11]:
vw = p.stage("final_views")

vw.run("work_mem", "SET work_mem = '1GB';")

vw.run("create_mv_owner_asset_weight", f"""
    DROP MATERIALIZED VIEW IF EXISTS {DST}.mv_owner_asset_weight;

    CREATE MATERIALIZED VIEW {DST}.mv_owner_asset_weight AS
    WITH h AS (
        SELECT ryear, rquarter, cik, cusip, value
        FROM {DST}.holdings_clean
        WHERE value > 0
    )
    SELECT ryear,
           rquarter,
           cik   AS owner,
           cusip AS asset,
           (value / SUM(value) OVER (PARTITION BY cik, ryear, rquarter))::double precision AS weight
    FROM h
    ORDER BY ryear, rquarter, cik, cusip      -- stores rows in quarter order, so per-quarter reads touch few pages
    WITH DATA;
""", expected=67_649_392, note="one row per holdings_clean row with value > 0")

vw.run("uq_mv_oaw_quarter_owner_asset", f"""
    CREATE UNIQUE INDEX uq_mv_oaw_quarter_owner_asset
        ON {DST}.mv_owner_asset_weight (ryear, rquarter, owner, asset);
""")
vw.run("idx_mv_oaw_asset", f"""
    CREATE INDEX idx_mv_oaw_asset ON {DST}.mv_owner_asset_weight (asset, ryear, rquarter);
""")

vw.run("reset_work_mem", "RESET work_mem;")
vw.run("analyze_mv_owner_asset_weight", f"ANALYZE {DST}.mv_owner_asset_weight;")

vw.run("mv_oaw_weight_sums", f"""
    SELECT count(*)                                  AS owner_quarters,
           count(*) FILTER (WHERE abs(w - 1) > 1e-6) AS not_summing_to_1,
           min(w)                                    AS min_sum,
           max(w)                                    AS max_sum
    FROM (SELECT sum(weight) AS w
          FROM {DST}.mv_owner_asset_weight
          GROUP BY ryear, rquarter, owner) s;
""", note="~1-2 min. Expected: not_summing_to_1 = 0")

✅ **[final_views] work_mem**: `ok` | `SET` | 0.0s

✅ **[final_views] create_mv_owner_asset_weight**: `ok` | `SELECT 67299418` | rows: **67,299,418** | expected: 67,649,392 (diff -349,974) | 243.4s

> one row per holdings_clean row with value > 0

NOTICES: NOTICE:  materialized view "mv_owner_asset_weight" does not exist, skipping

✅ **[final_views] uq_mv_oaw_quarter_owner_asset**: `ok` | `CREATE INDEX` | 223.7s

✅ **[final_views] idx_mv_oaw_asset**: `ok` | `CREATE INDEX` | 104.9s

...

✅ **[final_views] analyze_mv_owner_asset_weight**: `ok` | `ANALYZE` | 1.2s

✅ **[final_views] mv_oaw_weight_sums**: `ok` | `SELECT 1` | rows: **1** | 14.9s

> ~1-2 min. Expected: not_summing_to_1 = 0

,owner_quarters,not_summing_to_1,min_sum,max_sum
0,276848,0,1.0,1.0


,owner_quarters,not_summing_to_1,min_sum,max_sum
0,276848,0,1.0,1.0


## 7. Checks

### Row counts against `clean_up_1`

In [12]:
chk = p.stage("final_checks")

chk.run("row_counts", f"""
    SELECT t.table_name,
           (xpath('/row/n/text()', query_to_xml(format('SELECT count(*) AS n FROM {SRC}.%I', t.table_name),
                                                false, true, '')))[1]::text::bigint AS {SRC},
           (xpath('/row/n/text()', query_to_xml(format('SELECT count(*) AS n FROM {DST}.%I', t.table_name),
                                                false, true, '')))[1]::text::bigint AS {DST}
    FROM unnest(ARRAY{TABLES!r}) AS t(table_name);
""", note="equal except holdings_clean (holdings whose CUSIP is not in asset and earlier reports of a quarter dropped, sub-positions summed)")

✅ **[final_checks] row_counts**: `ok` | `SELECT 5` | rows: **5** | 5.9s

> equal except holdings_clean (holdings whose CUSIP is not in asset and earlier reports of a quarter dropped, sub-positions summed)

,table_name,clean_up_1,graph_db
0,ff_industry,209,209
1,sic_ff,59627,59627
2,edgar_company,88009,88009
3,asset,40593,40593
4,holdings_clean,97814731,67299418


,table_name,clean_up_1,graph_db
0,ff_industry,209,209
1,sic_ff,59627,59627
2,edgar_company,88009,88009
3,asset,40593,40593
4,holdings_clean,97814731,67299418


### Dropped holdings and owners

What was left out of `graph_db.holdings_clean`, and the owners affected by it. `holdings_totals`: lines of the
earlier reports of a quarter (not kept), and the `value` / `shares` totals of the kept lines against `graph_db`
(equal: summing the sub-positions loses nothing).

In [13]:
chk.run("dropped_holdings", f"""
    SELECT count(*)                       AS holdings_rows,
           count(DISTINCT h.cusip9)        AS cusips,
           count(DISTINCT h.cik)          AS owners,
           round(sum(h.value))            AS value,
           round(100.0 * sum(h.value) / (SELECT sum(value) FROM {SRC}.holdings_clean), 4) AS value_pct
    FROM {SRC}.holdings_clean h
    WHERE NOT EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9);
""", note="scans clean_up_1.holdings_clean (~2-3 min). Previous test: 37,081 rows | 7,803 CUSIPs | 1,544 owners | "
          "0.013% of value")

chk.run("owners", f"""
    SELECT count(*) FILTER (WHERE d.is_owner)                    AS owners_graph_db,
           count(*) FILTER (WHERE s.is_owner)                    AS owners_{SRC},
           count(*) FILTER (WHERE s.is_owner AND NOT d.is_owner) AS no_longer_owner,
           count(*) FILTER (WHERE d.is_13f_filer)               AS f13_filers
    FROM {DST}.edgar_company d
    JOIN {SRC}.edgar_company s USING (cik);
""")

chk.run("holdings_totals", f"""
    WITH latest AS (
        SELECT h.cik, max(h.rdate) AS rdate
        FROM {SRC}.holdings_clean h
        WHERE EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9)
        GROUP BY h.cik, date_trunc('quarter', h.rdate)
    ),
    s AS (
        SELECT count(*)      FILTER (WHERE l.cik IS NULL)     AS rows_earlier_reports,
               sum(h.value)  FILTER (WHERE l.cik IS NULL)     AS value_earlier_reports,
               count(*)      FILTER (WHERE l.cik IS NOT NULL) AS rows_kept,
               sum(h.value)  FILTER (WHERE l.cik IS NOT NULL) AS value,
               sum(h.shares) FILTER (WHERE l.cik IS NOT NULL) AS shares
        FROM {SRC}.holdings_clean h
        LEFT JOIN latest l USING (cik, rdate)
        WHERE EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9)
    ),
    d AS (
        SELECT count(*) AS rows, sum(value) AS value, sum(shares) AS shares
        FROM {DST}.holdings_clean
    )
    SELECT s.rows_earlier_reports,
           round(100.0 * s.value_earlier_reports / (s.value + s.value_earlier_reports), 4) AS value_pct_earlier_reports,
           s.rows_kept AS {SRC}_rows_kept, d.rows AS {DST}_rows,
           s.value = d.value AS value_equal, s.shares = d.shares AS shares_equal
    FROM s, d;
""", note="~4 min. Previous test: 165,941 lines in earlier reports (0.1197% of value) | "
          "101,611,848 lines kept -> 67,649,392 rows | totals equal")

✅ **[final_checks] dropped_holdings**: `ok` | `SELECT 1` | rows: **1** | 230.1s

> scans clean_up_1.holdings_clean (~2-3 min). Previous test: 37,081 rows | 7,803 CUSIPs | 1,544 owners | 0.013% of value

,holdings_rows,cusips,owners,value,value_pct
0,36928,7759,1518,253121205,0.0190


✅ **[final_checks] owners**: `ok` | `SELECT 1` | rows: **1** | 0.1s

,owners_graph_db,owners_clean_up_1,no_longer_owner,f13_filers
0,10681,10681,0,20108


✅ **[final_checks] holdings_totals**: `ok` | `SELECT 1` | rows: **1** | 431.9s

> ~4 min. Previous test: 165,941 lines in earlier reports (0.1197% of value) | 101,611,848 lines kept -> 67,649,392 rows | totals equal

,rows_earlier_reports,value_pct_earlier_reports,clean_up_1_rows_kept,graph_db_rows,value_equal,shares_equal
0,160653,0.1240,97617150,67299418,True,True


,rows_earlier_reports,value_pct_earlier_reports,clean_up_1_rows_kept,graph_db_rows,value_equal,shares_equal
0,160653,0.1240,97617150,67299418,True,True


### Report dates within a quarter

Why section 5 keeps only the latest report of an owner in a quarter. `graph_db` has no `rdate`, so this reads
`clean_up_1.holdings_clean` (holdings whose CUSIP is in `asset`). Read-only, needs only section 0 (own phase label
`rdate_analysis`), ~5 minutes.

**Distance from the quarter end.** `days_before` = quarter end − `rdate`, in week buckets; the 2nd / 1st month of
the quarter are one bucket each. `holdings_lines` are `clean_up_1` lines (before summing the sub-positions);
`pct_off_quarter_end` = share of the lines that are not on a quarter end.

In [14]:
rd = p.stage("rdate_analysis")

rd.run("work_mem", "SET work_mem = '1GB';")

rd.run("rdate_buckets", f"""
    WITH h AS (
        SELECT h.cik, h.rdate,
               (date_trunc('quarter', h.rdate) + interval '3 months - 1 day')::date - h.rdate AS days_before
        FROM {SRC}.holdings_clean h
        WHERE EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9)
    ),
    b AS (
        SELECT CASE WHEN days_before = 0   THEN '0 quarter end'
                    WHEN days_before <= 7  THEN '1-7 days before'
                    WHEN days_before <= 14 THEN '8-14 days before'
                    WHEN days_before <= 21 THEN '15-21 days before'
                    WHEN days_before <= 28 THEN '22-28 days before'
                    WHEN days_before <= 61 THEN '29-61 (2nd month of quarter)'
                    ELSE                        '62+ (1st month of quarter)' END AS rdate_bucket,
               min(days_before)             AS min_days,
               max(days_before)             AS max_days,
               count(*)                     AS holdings_lines,
               count(DISTINCT (cik, rdate)) AS filings,
               count(DISTINCT cik)          AS owners,
               count(DISTINCT rdate)        AS rdates
        FROM h
        GROUP BY 1
    )
    SELECT rdate_bucket, min_days, max_days, holdings_lines,
           round(100.0 * holdings_lines / sum(holdings_lines) OVER (), 3) AS pct_all,
           CASE WHEN min_days > 0
                THEN round(100.0 * holdings_lines / sum(holdings_lines) FILTER (WHERE min_days > 0) OVER (), 2)
           END AS pct_off_quarter_end,
           filings, owners, rdates
    FROM b
    ORDER BY min_days;
""", note="~3 min. Previous test (graph_db positions): 99.04% on a quarter end; of the rest 80.9% 1-7 days "
          "before (mostly 28th-30th), 11.9% in the 2nd month, 2.9% in the 1st month")

✅ **[rdate_analysis] work_mem**: `ok` | `SET` | 0.0s

✅ **[rdate_analysis] rdate_buckets**: `ok` | `SELECT 7` | rows: **7** | 270.8s

> ~3 min. Previous test (graph_db positions): 99.04% on a quarter end; of the rest 80.9% 1-7 days before (mostly 28th-30th), 11.9% in the 2nd month, 2.9% in the 1st month

,rdate_bucket,min_days,max_days,holdings_lines,pct_all,pct_off_quarter_end,filings,owners,rdates
0,0 quarter end,0,0,96551310,98.746,None,275315,10673,88
1,1-7 days before,1,7,1060940,1.085,86.50,1259,549,55
2,8-14 days before,10,14,6574,0.007,0.54,36,36,30
3,15-21 days before,15,21,11434,0.012,0.93,25,24,20
4,22-28 days before,22,28,17381,0.018,1.42,62,59,34
5,29-61 (2nd month of quarter),29,61,103445,0.106,8.43,539,362,251
6,62+ (1st month of quarter),62,91,26719,0.027,2.18,150,104,133


,rdate_bucket,min_days,max_days,holdings_lines,pct_all,pct_off_quarter_end,filings,owners,rdates
0,0 quarter end,0,0,96551310,98.746,None,275315,10673,88
1,1-7 days before,1,7,1060940,1.085,86.50,1259,549,55
2,8-14 days before,10,14,6574,0.007,0.54,36,36,30
3,15-21 days before,15,21,11434,0.012,0.93,25,24,20
4,22-28 days before,22,28,17381,0.018,1.42,62,59,34
5,29-61 (2nd month of quarter),29,61,103445,0.106,8.43,539,362,251
6,62+ (1st month of quarter),62,91,26719,0.027,2.18,150,104,133


**Collisions.** (`cik`, quarter, `cusip`) groups with more than one `rdate`: the owner filed two (or more) reports
with different report dates in one quarter. `rdate_holding` holds the positions (sub-positions summed, as in
`graph_db`) of the owner-quarters that have an off-quarter-end `rdate` (only those can have two). Each collision is
one row with the values per `rdate` (arrays in `rdate` order), classified as

* `identical`: same `shares` and `value` (the same report twice, e.g. 2007-03-30 and 2007-03-31)
* `same shares, other value`: same position, priced at the two dates
* `different shares`: the position changed between the two report dates

In [15]:
rd.run("rdate_holding_build", f"""
    DROP TABLE IF EXISTS pg_temp.rdate_holding;
    CREATE TEMP TABLE rdate_holding AS
    SELECT h.cik, h.rdate, h.cusip9 AS cusip,
           max(h.fdate) AS fdate, sum(h.value) AS value, sum(h.shares) AS shares
    FROM {SRC}.holdings_clean h
    WHERE EXISTS (SELECT 1 FROM {DST}.asset a WHERE a.cusip = h.cusip9)
      AND (h.cik, date_trunc('quarter', h.rdate)) IN (
              SELECT cik, date_trunc('quarter', rdate)
              FROM {SRC}.holdings_clean
              WHERE rdate <> (date_trunc('quarter', rdate) + interval '3 months - 1 day')::date)
    GROUP BY h.cik, h.rdate, h.cusip9;
""", note="~2 min. Session temp table, used by the next cells")

rd.run("rdate_collisions_build", """
    DROP TABLE IF EXISTS pg_temp.rdate_collision;
    CREATE TEMP TABLE rdate_collision AS
    SELECT cik,
           to_char(date_trunc('quarter', rdate), 'YYYY"Q"Q')  AS quarter,
           cusip,
           count(*)                                          AS n,
           array_agg(rdate  ORDER BY rdate)                  AS rdates,
           array_agg(fdate  ORDER BY rdate)                  AS fdates,
           array_agg(shares ORDER BY rdate)                  AS shares,
           array_agg(value  ORDER BY rdate)                  AS values,
           CASE WHEN min(shares) = max(shares) AND min(value) = max(value) THEN 'identical'
                WHEN min(shares) = max(shares)                             THEN 'same shares, other value'
                ELSE 'different shares' END                  AS kind
    FROM pg_temp.rdate_holding
    GROUP BY cik, date_trunc('quarter', rdate), cusip
    HAVING count(*) > 1;
""", expected=77_070)

rd.run("rdate_collisions_by_kind", """
    SELECT kind,
           count(*)                                           AS groups,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           count(DISTINCT cik)                                AS owners,
           count(DISTINCT (cik, quarter))                     AS owner_quarters
    FROM pg_temp.rdate_collision
    GROUP BY 1
    ORDER BY 2 DESC;
""", note="Previous test: different shares 50.7% | identical 38.6% | same shares, other value 10.7%")

✅ **[rdate_analysis] rdate_holding_build**: `ok` | `SELECT 794553` | rows: **794,553** | 237.3s

> ~2 min. Session temp table, used by the next cells

NOTICES: NOTICE:  table "rdate_holding" does not exist, skipping

✅ **[rdate_analysis] rdate_collisions_build**: `ok` | `SELECT 76050` | rows: **76,050** | expected: 77,070 (diff -1,020) | 0.7s

> NOTICES: NOTICE:  table "rdate_collision" does not exist, skipping

✅ **[rdate_analysis] rdate_collisions_by_kind**: `ok` | `SELECT 3` | rows: **3** | 0.0s

> Previous test: different shares 50.7% | identical 38.6% | same shares, other value 10.7%

,kind,groups,pct,owners,owner_quarters
0,different shares,38138,50.1,387,461
1,identical,29738,39.1,158,189
2,"same shares, other value",8174,10.7,311,367


,kind,groups,pct,owners,owner_quarters
0,different shares,38138,50.1,387,461
1,identical,29738,39.1,158,189
2,"same shares, other value",8174,10.7,311,367


Examples of each kind (a fixed pseudo-random pick: `md5` of the key, so a rerun shows the same rows).

In [16]:
N_EXAMPLES = 8

for kind in ["identical", "same shares, other value", "different shares"]:
    rd.run(f"rdate_examples_{kind.split(',')[0].replace(' ', '_')}", f"""
        SELECT cik, quarter, cusip, rdates, fdates, shares, values
        FROM pg_temp.rdate_collision
        WHERE kind = '{kind}'
        ORDER BY md5(cik || quarter || cusip)
        LIMIT {N_EXAMPLES};
    """, note=kind)

✅ **[rdate_analysis] rdate_examples_identical**: `ok` | `SELECT 8` | rows: **8** | 0.0s

> identical

,cik,quarter,cusip,rdates,fdates,shares,values
0,948046,2007Q1,78412D109,"[2007-03-30, 2007-03-31]","[2007-05-15, 2007-05-15]","[3783.000, 3783.000]","[28.00000, 28.00000]"
1,312069,2008Q3,826565103,"[2008-07-31, 2008-09-30]","[2008-08-12, 2008-11-14]","[628.000, 628.000]","[9.00000, 9.00000]"
2,1324943,2011Q3,844030106,"[2011-08-11, 2011-09-30]","[2011-08-12, 2011-11-14]","[735.000, 735.000]","[30.00000, 30.00000]"
3,948046,2007Q1,281347104,"[2007-03-30, 2007-03-31]","[2007-05-15, 2007-05-15]","[458025.000, 458025.000]","[12000.00000, 12000.00000]"
4,1315421,2007Q1,852061100,"[2007-03-30, 2007-03-31]","[2007-11-08, 2007-05-15]","[29594.000, 29594.000]","[561.00000, 561.00000]"
5,1092307,2000Q4,285661104,"[2000-12-29, 2000-12-31]","[2001-03-16, 2001-03-01]","[4454323.000, 4454323.000]","[257235.00000, 257235.00000]"
6,36966,2003Q1,337477103,"[2003-01-31, 2003-03-31]","[2003-10-27, 2003-04-22]","[10190.000, 10190.000]","[399.75200, 399.75200]"
7,948046,2007Q1,029429107,"[2007-03-30, 2007-03-31]","[2007-05-15, 2007-05-15]","[10725.000, 10725.000]","[564.00000, 564.00000]"


✅ **[rdate_analysis] rdate_examples_same_shares**: `ok` | `SELECT 8` | rows: **8** | 0.0s

> same shares, other value

,cik,quarter,cusip,rdates,fdates,shares,values
0,863749,2005Q3,601073109,"[2005-07-30, 2005-09-30]","[2005-08-15, 2005-10-25]","[19550.000, 19550.000]","[1109.00000, 1229.00000]"
1,1324943,2011Q3,723484101,"[2011-08-11, 2011-09-30]","[2011-08-12, 2011-11-14]","[1300.000, 1300.000]","[58.00000, 56.00000]"
2,905571,2009Q4,049513104,"[2009-11-13, 2009-12-31]","[2009-11-13, 2010-02-16]","[17300.000, 17300.000]","[72.00000, 80.00000]"
3,905571,2005Q2,904784709,"[2005-05-11, 2005-06-30]","[2005-05-13, 2005-08-09]","[690.000, 690.000]","[47.00000, 45.00000]"
4,1135986,2010Q1,459902102,"[2010-02-16, 2010-03-31]","[2010-02-16, 2010-05-17]","[3000.000, 3000.000]","[57.00000, 56.00000]"
5,902367,1999Q3,313135303,"[1999-07-19, 1999-09-30]","[1999-07-20, 1999-11-12]","[15705.000, 15705.000]","[73.00000, 78.00000]"
6,1218100,2003Q4,962166104,"[2003-10-23, 2003-12-31]","[2003-10-23, 2004-01-29]","[5501.000, 5501.000]","[322.00000, 352.00000]"
7,60086,2007Q1,808541106,"[2007-02-13, 2007-03-31]","[2007-02-13, 2007-05-03]","[21875.000, 21875.000]","[570.00000, 544.00000]"


✅ **[rdate_analysis] rdate_examples_different_shares**: `ok` | `SELECT 8` | rows: **8** | 0.0s

> different shares

,cik,quarter,cusip,rdates,fdates,shares,values
0,1056053,2009Q4,38141G104,"[2009-10-30, 2009-12-31]","[2009-11-12, 2010-02-12]","[907449.000, 733729.000]","[167288.00000, 123883.00000]"
1,846788,2002Q2,404132102,"[2002-04-30, 2002-06-30]","[2002-05-06, 2002-07-30]","[149968.000, 148668.000]","[3899.00000, 3917.00000]"
2,806085,2001Q4,023551104,"[2001-11-15, 2001-12-13]","[2001-11-15, 2002-02-14]","[11149.000, 9928.000]","[707.00000, 620.00000]"
3,64908,2006Q4,499064103,"[2006-10-25, 2006-12-31]","[2006-10-25, 2007-01-23]","[76750.000, 79450.000]","[1301.00000, 1355.00000]"
4,316471,2010Q1,53217V109,"[2010-02-12, 2010-03-31]","[2010-02-12, 2010-05-17]","[366921.000, 365367.000]","[19161.00000, 19098.00000]"
5,1352543,2010Q4,911163103,"[2010-10-29, 2010-12-31]","[2010-11-02, 2011-02-04]","[659345.000, 483303.000]","[21850.00000, 17727.00000]"
6,1076598,1999Q2,191216100,"[1999-05-24, 1999-06-30]","[1999-05-25, 1999-07-30]","[1106639.000, 1289353.000]","[67920.00000, 79939.00000]"
7,895421,2011Q3,67063C106,"[2011-08-03, 2011-09-30]","[2011-08-12, 2011-11-15]","[29419.000, 25506.000]","[378.00000, 332.00000]"


**Per report.** The same owner-quarters as whole reports: one row per `rdate`, `rk` = 1 for the latest report of the
quarter (the one section 5 keeps). `positions_in_earlier_reports` = `graph_db` rows the rule leaves out.

In [17]:
rd.run("rdate_filings_build", """
    DROP TABLE IF EXISTS pg_temp.rdate_filing;
    CREATE TEMP TABLE rdate_filing AS
    SELECT cik,
           date_trunc('quarter', rdate)::date                                                AS q,
           rdate,
           max(fdate)                                                                        AS fdate,
           count(*)                                                                          AS positions,
           round(sum(value))                                                                 AS value,
           rank()   OVER (PARTITION BY cik, date_trunc('quarter', rdate) ORDER BY rdate DESC) AS rk,
           count(*) OVER (PARTITION BY cik, date_trunc('quarter', rdate))                     AS reports_in_q
    FROM pg_temp.rdate_holding
    GROUP BY cik, date_trunc('quarter', rdate), rdate;
""")

rd.run("rdate_filings_summary", """
    SELECT count(DISTINCT (cik, q))                                                   AS owner_quarters,
           count(DISTINCT cik)                                                        AS owners,
           sum(positions) FILTER (WHERE rk > 1)                                       AS positions_in_earlier_reports,
           count(*) FILTER (WHERE rk = 1
                              AND rdate = (q + interval '3 months - 1 day')::date)    AS latest_is_quarter_end,
           max(reports_in_q)                                                          AS max_reports_in_quarter
    FROM pg_temp.rdate_filing
    WHERE reports_in_q > 1;
""", note="Previous test: 528 owner-quarters (433 owners), 89,998 positions in earlier reports, "
          "latest = quarter end in 519")

rd.run("rdate_filings_examples", f"""
    SELECT cik, to_char(q, 'YYYY"Q"Q') AS quarter, rdate, fdate, positions, value, rk
    FROM pg_temp.rdate_filing
    WHERE (cik, q) IN (SELECT cik, q
                       FROM (SELECT DISTINCT cik, q FROM pg_temp.rdate_filing WHERE reports_in_q > 1) d
                       ORDER BY md5(cik::text || q::text)
                       LIMIT {N_EXAMPLES})
    ORDER BY cik, q, rdate;
""")

rd.run("drop_rdate_temp", "DROP TABLE pg_temp.rdate_holding, pg_temp.rdate_collision, pg_temp.rdate_filing;")
rd.run("reset_work_mem", "RESET work_mem;")

✅ **[rdate_analysis] rdate_filings_build**: `ok` | `SELECT 2588` | rows: **2,588** | 0.2s

> NOTICES: NOTICE:  table "rdate_filing" does not exist, skipping

✅ **[rdate_analysis] rdate_filings_summary**: `ok` | `SELECT 1` | rows: **1** | 0.0s

> Previous test: 528 owner-quarters (433 owners), 89,998 positions in earlier reports, latest = quarter end in 519

,owner_quarters,owners,positions_in_earlier_reports,latest_is_quarter_end,max_reports_in_quarter
0,526,431,89040,517,4


✅ **[rdate_analysis] rdate_filings_examples**: `ok` | `SELECT 16` | rows: **16** | 0.0s

,cik,quarter,rdate,fdate,positions,value,rk
0,919500,2005Q2,2005-04-27,2005-04-28,1,17978,2
1,919500,2005Q2,2005-06-30,2005-07-14,2,108046,1
2,931328,2002Q2,2002-04-30,2002-05-13,16,8766,2
3,931328,2002Q2,2002-06-30,2002-08-12,46,29632,1
4,1056866,2002Q2,2002-06-01,2002-10-21,211,72230,2
5,1056866,2002Q2,2002-06-30,2002-07-31,528,341130,1
6,1085041,2001Q2,2001-04-30,2001-05-04,37,31958,2
7,1085041,2001Q2,2001-06-30,2001-08-02,68,51849,1
8,1106059,2001Q2,2001-06-01,2001-08-22,11,2800,2
9,1106059,2001Q2,2001-06-30,2001-08-14,10,6700,1


✅ **[rdate_analysis] drop_rdate_temp**: `ok` | `DROP TABLE` | 0.0s

✅ **[rdate_analysis] reset_work_mem**: `ok` | `RESET` | 0.0s

""


### Keys and issuers

All foreign keys of `graph_db` (validated when they were created) and the asset → issuer coverage.

In [18]:
chk.run("foreign_keys", f"""
    SELECT conrelid::regclass AS table_name, conname, pg_get_constraintdef(oid) AS definition, convalidated
    FROM pg_constraint
    WHERE connamespace = '{DST}'::regnamespace AND contype = 'f'
    ORDER BY 1, 2;
""", note="expected 4: sic_ff -> ff_industry, asset -> edgar_company, "
          "holdings_clean -> asset, holdings_clean -> edgar_company")

chk.run("asset_issuers", f"""
    SELECT industry_match, count(*) AS assets, count(issuer_cik) AS with_issuer,
           count(*) FILTER (WHERE EXISTS (SELECT 1 FROM {DST}.holdings_clean h WHERE h.cusip = a.cusip)) AS held
    FROM {DST}.asset a
    GROUP BY 1
    ORDER BY 2 DESC;
""")

✅ **[final_checks] foreign_keys**: `ok` | `SELECT 4` | rows: **4** | 0.0s

> expected 4: sic_ff -> ff_industry, asset -> edgar_company, holdings_clean -> asset, holdings_clean -> edgar_company

,table_name,conname,definition,convalidated
0,graph_db.sic_ff,sic_ff_scheme_industry_number_fkey,"FOREIGN KEY (scheme, industry_number) REFERENC...",True
1,graph_db.asset,asset_issuer_cik_fkey,FOREIGN KEY (issuer_cik) REFERENCES graph_db.e...,True
2,graph_db.holdings_clean,fk_holdings_asset,FOREIGN KEY (cusip) REFERENCES graph_db.asset(...,True
3,graph_db.holdings_clean,fk_holdings_company,FOREIGN KEY (cik) REFERENCES graph_db.edgar_co...,True


✅ **[final_checks] asset_issuers**: `ok` | `SELECT 6` | rows: **6** | 2.7s

,industry_match,assets,with_issuer,held
0,name,25461,25461,15022
1,unmatched,8251,0,6432
2,name+ticker,4308,4308,2908
3,name+same_industry,1413,1413,907
4,ambiguous,662,0,380
5,name+period,498,498,239


,industry_match,assets,with_issuer,held
0,name,25461,25461,15022
1,unmatched,8251,0,6432
2,name+ticker,4308,4308,2908
3,name+same_industry,1413,1413,907
4,ambiguous,662,0,380
5,name+period,498,498,239


In [19]:
for t in TABLES + VIEWS:
    chk.table_size(f"{DST}.{t}", t)

✅ **[final_checks] ff_industry_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,64 kB,16 kB,16 kB


✅ **[final_checks] sic_ff_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,4912 kB,2584 kB,2296 kB


✅ **[final_checks] edgar_company_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

...

,total,heap,indexes
0,6072 kB,3816 kB,2216 kB


✅ **[final_checks] holdings_clean_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,9866 MB,4915 MB,4950 MB


✅ **[final_checks] mv_owner_asset_weight_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,6947 MB,3867 MB,3080 MB


## 8. Run log analytics

Reads `steps.jsonl` from **all** runs in `logs/finalisation/`.

In [20]:
logs = load_logs(LOG_ROOT)
steps_table(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
164,1,connection,connection_check_server,ok,None,NaN,NaN,NaN,0.020
165,2,connection,connection_check_schemas,ok,None,NaN,NaN,NaN,NaN
166,3,connection,connection_check_tables,ok,None,NaN,NaN,NaN,NaN
167,4,final_schema,create_schema,ok,CREATE SCHEMA,-1.0,NaN,NaN,0.001
168,5,final_schema,drop_tables,ok,DROP TABLE,-1.0,NaN,NaN,0.051
...,...,...,...,...,...,...,...,...,...
221,58,final_checks,sic_ff_size,ok,SELECT 1,1.0,NaN,NaN,0.000
222,59,final_checks,edgar_company_size,ok,SELECT 1,1.0,NaN,NaN,0.000
223,60,final_checks,asset_size,ok,SELECT 1,1.0,NaN,NaN,0.000
224,61,final_checks,holdings_clean_size,ok,SELECT 1,1.0,NaN,NaN,0.000


In [21]:
# Row counts that differ from the previous run
expected_mismatches(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
178,15,final_holdings,latest_report,ok,SELECT 276848,276848.0,276423.0,425.0,79.343
179,16,final_holdings,insert_holdings,ok,INSERT 0 67299418,67299418.0,67649392.0,-349974.0,382.351
195,32,final_views,create_mv_owner_asset_weight,ok,SELECT 67299418,67299418.0,67649392.0,-349974.0,243.447
208,45,rdate_analysis,rdate_collisions_build,ok,SELECT 76050,76050.0,77070.0,-1020.0,0.736


In [22]:
phase_durations(logs, p.run_id)

,seconds
phase,
connection,0.020
final_schema,0.052
final_industry,0.269
final_company,0.191
final_asset,0.205
final_holdings,840.271
final_views,588.187
final_checks,670.713
rdate_analysis,509.129


In [23]:
compare_runs(logs)

rowcount                                                                                      duration_s                                  \
run_id                                       20261011_204130 20261011_204259 20261012_151310 20261014_114036 20261014_191117 20261029_160620 20261011_204130 20261011_204259 20261012_151310   
phase          step                                                                                                                                                                            
connection     connection_check_server                   NaN             NaN             NaN             NaN             NaN             NaN           0.029           0.018           0.019   
final_schema   create_schema                            -1.0            -1.0            -1.0            -1.0             NaN            -1.0           0.006           0.000           0.000   
               drop_tables                              -1.0            -1.0            -1.0            -1.0             NaN            -1.0           0.000           0.011           0.025   
final_industry create_ff_industry                      209.0           209.0           209.0           209.0             NaN           209.0           0.022           0.011           0.016   
               create_sic_ff                         59627.0         59627.0         59627.0         59627.0             NaN         59627.0           0.269           0.222           0.211   
...                                                      ...             ...             ...             ...             ...             ...             ...             ...             ...   
final_views    create_mv_owner_asset_weight              NaN             NaN             NaN             NaN      67649392.0      67299418.0             NaN             NaN             NaN   
               uq_mv_oaw_quarter_owner_asset             NaN             NaN             NaN             NaN            -1.0            -1.0             NaN             NaN             NaN   
               idx_mv_oaw_asset                          NaN             NaN             NaN             NaN            -1.0            -1.0             NaN             NaN             NaN   
               analyze_mv_owner_asset_weight             NaN             NaN             NaN             NaN            -1.0            -1.0             NaN             NaN             NaN   
               mv_oaw_weight_sums                        NaN             NaN             NaN             NaN             1.0             1.0             NaN             NaN             NaN   

                                                                                              
run_id                                       20261014_114036 20261014_191117 20261029_160620  
phase          step                                                                           
connection     connection_check_server                 0.016             NaN           0.020  
final_schema   create_schema                           0.000             NaN           0.001  
               drop_tables                             0.125             NaN           0.051  
final_industry create_ff_industry                      0.015             NaN           0.016  
               create_sic_ff                           0.234             NaN           0.229  
...                                                      ...             ...             ...  
final_views    create_mv_owner_asset_weight              NaN         186.840         243.447  
               uq_mv_oaw_quarter_owner_asset             NaN         188.179         223.679  
               idx_mv_oaw_asset                          NaN          81.791         104.916  
               analyze_mv_owner_asset_weight             NaN           1.062           1.203  
               mv_oaw_weight_sums                        NaN          11.745          14.942  

[70 rows x 12 columns]

In [24]:
p.close()

Run log: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/logs/finalisation/20261029_160620/steps.jsonl
